In [0]:
from pyspark.sql.functions import *


In [0]:
gold_fact_orders = spark.read.table("olist.gold.fact_order_items")

In [0]:
# Total orders by per customer

total_orders_customer = gold_fact_orders.filter(col("customer_id").isNotNull()).groupBy("customer_id").agg(count("order_id").alias("total_orders")).orderBy(desc("total_orders"))


Most sold product

In [0]:
top_selling_product = gold_fact_orders.groupBy("product_id").agg(sum("price").alias("total_revenue")).orderBy(desc("total_revenue")).limit(10)


Top Customer by spending

In [0]:
top_customer = gold_fact_orders.filter(col("customer_id").isNotNull()).groupBy("customer_id").agg(sum("price").alias("total_spent")).orderBy(desc("total_spent"))


average review score per seller

In [0]:
avg_review_seller = gold_fact_orders.groupBy("seller_id").agg(avg("review_score").alias("avg_review")).orderBy(desc("avg_review"))

Total Revenue & Average order value (ADV) per customer

In [0]:
customer_spending_df = gold_fact_orders.filter(col("customer_id").isNotNull()).groupBy("customer_id").agg(count("order_id").alias("total_orders"), sum("price").alias("total_spent"),round(avg("price"),2).alias("AOV")).orderBy(desc("total_spent"))


Seller metrics (Revenue,avg review,order_count)

In [0]:
seller_df = gold_fact_orders.groupBy("seller_id").agg(count("order_id").alias("total_orders"), sum("price").alias("total_revenue"),round(avg("price"),2).alias("AOV"),round(stddev("price"),2).alias("price_variablity")).orderBy(desc("total_revenue"))


# Monthly revenue trend

In [0]:
monthly_revenue = gold_fact_orders.filter(
    col("order_purchase_timestamp").isNotNull()  # drop null rows
).groupBy(
    year("order_purchase_timestamp").alias("year"),
    month("order_purchase_timestamp").alias("month")
).agg(
    sum("price").alias("total_revenue"),
    count("order_id").alias("total_orders"),
    avg("price").alias("avg_order_value")
).orderBy(desc("total_revenue"))

In [0]:
gold_agg_tables = {
    "agg_total_orders_customer"  : total_orders_customer,
    "agg_top_selling_product"    : top_selling_product,
    "agg_top_customer"           : top_customer,
    "agg_avg_review_seller"      : avg_review_seller,
    "agg_customer_spending"      : customer_spending_df,
    "agg_seller_scorecard"       : seller_df,
    "agg_monthly_revenue"        : monthly_revenue,
}

for table_name, df in gold_agg_tables.items():
    print(f"Writing {table_name}...")
    df.write.format("delta") \
      .mode("overwrite") \
      .saveAsTable(f"olist.gold.{table_name}")
    print(f"{table_name} saved")

In [0]:
optimize_configs = {
    "agg_total_orders_customer" : "customer_id",
    "agg_top_selling_product"   : "product_id",
    "agg_top_customer"          : "customer_id",
    "agg_avg_review_seller"     : "seller_id",
    "agg_customer_spending"     : "customer_id",
    "agg_seller_scorecard"      : "seller_id",
    "agg_monthly_revenue"       : "year, month",
}

for table_name, zorder_col in optimize_configs.items():
    print(f"Optimizing {table_name}...")
    spark.sql(f"OPTIMIZE olist.gold.{table_name} ZORDER BY ({zorder_col})")
    print(f"{table_name} optimized")